# 03 – Model Training: Autoencoder för bedrägeridetektion

**Föregående steg:** `02_data_preparation.ipynb` sparade rensade, skalade och uppdelade data till `../data/processed/splits.npz` samt en fitted scaler till `../models/scaler.pkl`.

**Den här notebooken:**
1. Läser in `X_train` och `X_val` (endast legitima transaktioner)
2. Bygger en Dense-autoencoder (samma arkitekturmönster som i kursmaterialet)
3. Tränar modellen att rekonstruera legitima transaktioner
4. Bestämmer ett tröskelvärde för anomali-flaggning utifrån rekonstruktionsfelet på validation-setet (95:e percentilen)
5. Sparar modell och tröskel för användning i `04_evaluation.ipynb`

**Viktigt:** `X_test`/`y_test` i `splits.npz` rörs inte här — testsetet ska förbli helt orört fram till den slutgiltiga utvärderingen.

In [ ]:
import numpy as np
import tensorflow as tf

RANDOM_STATE = 42

# Tvinga enkeltrådig körning för fullt deterministisk CPU-exekvering.
# Måste sättas innan någon annan TensorFlow-operation körs, annars har det ingen effekt.
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
tf.config.experimental.enable_op_determinism()

from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import json
import os

os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)

print("TensorFlow version:", tf.__version__)
print("GPUs tillgängliga:", len(tf.config.list_physical_devices('GPU')))


## Läs in förberedda data

In [ ]:
data = np.load('../data/processed/splits.npz')

X_train = data['X_train']
X_val = data['X_val']

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)


**Vad ser vi?**

`X_train`: (181 281, 30), `X_val`: (45 321, 30) — matchar exakt det som sparades i `02_data_preparation.ipynb`. (TensorFlow 2.21.0 körs på CPU eftersom native GPU-stöd inte finns för TF ≥ 2.11 på Windows utan WSL2 — helt förväntat och inget problem för en modell av den här storleken.)

## Bygg autoencoder-arkitekturen

Samma grundmönster som i kursmaterialet: en symmetrisk Dense-autoencoder som gradvis komprimerar de 30 features till en flaskhals och sedan rekonstruerar dem igen.

- **Encoder:** 30 → 14 → 7
- **Bottleneck:** 7 (den komprimerade representationen)
- **Decoder:** 7 → 14 → 30
- **Output-aktivering:** `linear`, eftersom features är standardiserade och kan vara både positiva och negativa (till skillnad från t.ex. bildpixlar 0–1, där `sigmoid` används)
- **Loss:** MSE (Mean Squared Error) — mäter rekonstruktionsfelet, som senare används för att flagga anomalier

In [ ]:
input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(14, activation='relu')(input_layer)
encoded = Dense(7, activation='relu')(encoded)
decoded = Dense(14, activation='relu')(encoded)
output_layer = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.summary()


**Vad ser vi?**

Modellen har totalt **1101 parametrar (4,30 KB)** — mycket liten, vilket är rimligt för 30 tabellära features:
- `dense` (30→14): 434 parametrar
- `dense_1` (14→7, bottleneck): 105 parametrar
- `dense_2` (7→14): 112 parametrar
- `dense_3` (14→30): 450 parametrar

Den lilla parameterstorleken innebär snabb träning och låg risk för överanpassning givet 181 281 träningsrader — gott om data per parameter.

## Träna autoencodern

Modellen tränas att rekonstruera `X_train` (endast legitima transaktioner) — både input och target är samma data. `EarlyStopping` bevakar valideringsförlusten och återställer de bästa vikterna om träningen inte förbättras på 5 epoker i rad, vilket skyddar mot överanpassning utan att vi manuellt behöver gissa rätt antal epoker.

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history = autoencoder.fit(
    X_train, X_train,
    epochs=50,
    batch_size=64,
    shuffle=True,
    validation_data=(X_val, X_val),
    callbacks=[early_stop],
    verbose=1
)


**Vad ser vi?**

`EarlyStopping` triggade vid epok 47, återställde bästa vikter från **epok 42** (train loss 0,2768 / val loss 0,2801). Samma mönster som tidigare körningar: snabb konvergens, lång utplaning, stabil generalisering.

## Träningskurva

Vi plottar train- och validation-loss per epok för att bekräfta att modellen konvergerar utan att overfitta (dvs. att validation-loss inte börjar öka igen medan train-loss fortsätter sjunka).

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('MSE loss')
plt.title('Autoencoder Training')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/05_training_loss.png', dpi=150)
plt.show()


**Vad ser vi?**

Samma karaktäristiska form som i alla tidigare körningar: snabb nedgång de första ~10 epokerna, därefter utplaning runt 0,28–0,30. Train och validation följer varandra tätt, inga tecken på överanpassning.

## Bestäm tröskelvärde för anomali-flaggning

Vi beräknar rekonstruktionsfelet (MSE) för varje legitim transaktion i validation-setet, och sätter tröskeln vid **95:e percentilen** — samma metod som i kursexemplet. Det innebär att ~5 % av de legitima valideringstransaktionerna hamnar över tröskeln (förväntade falska positiva), medan transaktioner med betydligt högre rekonstruktionsfel flaggas som misstänkta.

In [ ]:
X_val_pred = autoencoder.predict(X_val, verbose=0)
mse_val = np.mean(np.square(X_val - X_val_pred), axis=1)

threshold = np.percentile(mse_val, 95)
print(f"Threshold (95:e percentilen): {threshold:.6f}")
print(f"Min/Max rekonstruktionsfel (validation): {mse_val.min():.4f} / {mse_val.max():.4f}")

# Fördelningen är extremt högerskevt (ett fåtal outliers sträcker sig långt bortom tröskeln),
# så vi visar både fullt intervall (log-skala) och en inzoomad vy där tröskeln faktiskt syns.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(mse_val, bins=100, color='steelblue')
axes[0].set_yscale('log')
axes[0].axvline(threshold, color='crimson', linestyle='--', label=f'Threshold = {threshold:.4f}')
axes[0].set_xlabel('Rekonstruktionsfel (MSE)')
axes[0].set_ylabel('Antal transaktioner (log-skala)')
axes[0].set_title('Fullt intervall')
axes[0].legend()

zoom_limit = threshold * 5
mse_val_zoom = mse_val[mse_val < zoom_limit]
axes[1].hist(mse_val_zoom, bins=100, color='steelblue')
axes[1].axvline(threshold, color='crimson', linestyle='--', label=f'Threshold = {threshold:.4f}')
axes[1].set_xlabel('Rekonstruktionsfel (MSE)')
axes[1].set_ylabel('Antal transaktioner')
axes[1].set_title(f'Inzoomat (MSE < {zoom_limit:.2f})')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/06_reconstruction_error_val.png', dpi=150)
plt.show()

with open('../models/threshold.json', 'w') as f:
    json.dump({'threshold_percentile': 95, 'threshold_value': float(threshold)}, f, indent=2)

print("Threshold sparad till ../models/threshold.json")


**Vad ser vi?**

Den inzoomade vyn visar majoriteten av legitima valideringstransaktioner med rekonstruktionsfel under ~0,5, topp runt 0,1–0,2, med tröskeln (0,6044) i den högra svansen — samma mönster som samtliga tidigare körningar.

Den fulla vyn bekräftar samma extremt högerskevda fördelning, med enstaka legitima outliers upp till 84,78 i rekonstruktionsfel.

## Spara modellen

Modellen sparas i Keras native-format (`.keras`) tillsammans med tröskeln, redo att laddas in i `04_evaluation.ipynb` utan att behöva träna om.

In [ ]:
autoencoder.save('../models/autoencoder.keras')
print("Modell sparad till ../models/autoencoder.keras")


## Sammanfattning

- **Arkitektur:** Dense-autoencoder 30 → 14 → 7 → 14 → 30, totalt 1101 parametrar (4,30 KB)
- **Träning:** EarlyStopping triggade vid epok 47, återställde bästa vikter från epok 42 (train loss 0,2768 / val loss 0,2801)
- **Tröskel:** 0,604441 (95:e percentilen). Rekonstruktionsfelet på validation spänner från 0,0108 till 84,78

**Not om reproducerbarhet:** Trots `enable_op_determinism()` och enkeltrådig CPU-körning varierar exakta loss-/tröskelvärden fortfarande något mellan körningar (tröskel har legat mellan ~0,60 och ~0,68 över flera försök). Resultaten är kvalitativt stabila (val loss konsekvent ~0,28–0,31), så det här är den slutgiltiga körningen vi går vidare med — notebooken körs inte om igen.

**Sparade artefakter (denna körning, används i nästa steg):**
- `../models/autoencoder.keras` — tränad autoencoder-modell
- `../models/threshold.json` — tröskelvärde (0,604441)

**Nästa steg:** `04_evaluation.ipynb` — ladda modell, scaler och tröskel, kör mot det orörda testsetet (`X_test`, `y_test`), och utvärdera med confusion matrix och classification report.